# Treino da U-Net v2 (lesões pequenas: 2000 px + perda de Tversky) — lesões periapicais (ML Odonto)

Treina a U-Net **só com o conjunto público de Hanói** (Mendeley, CC BY 4.0). Nenhuma radiografia
de paciente vem para o Colab: a avaliação e a aplicação rodam no computador local.

**Antes de começar:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.
Depois *Ambiente de execução → Executar tudo*. O Google vai pedir permissão para acessar o Drive
(os checkpoints ficam em `MyDrive/ML_Odonto_UNet_v2`, para não perder o treino se a sessão cair).

Tempo esperado: ~10 min de download e preparação + ~1h30-2h de treino.

**Se a sessão cair:** reconecte, rode as células de 1 a 6 de novo e, na célula 7, troque
`CONTINUAR = False` por `CONTINUAR = True`.

No fim, a última célula baixa `unet_lesoes.pt`: salve em `ML Odonto/Modelos/` com o nome `unet_lesoes_v2.pt`.

### 1. GPU

In [ ]:
!nvidia-smi -L

### 2. Google Drive (onde ficam os checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAIDA = '/content/drive/MyDrive/ML_Odonto_UNet_v2'
!mkdir -p "{SAIDA}" /content/scripts

### 3–5. Código do projeto (gerado a partir de `scripts/`)

In [ ]:
%%writefile /content/scripts/comum.py
"""Peças compartilhadas: carregamento, normalização, atributos por pixel e métricas."""
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import hessian_matrix, hessian_matrix_eigvals

RAIZ = Path(__file__).resolve().parent.parent
NOSSAS = RAIZ / "Treinamento" / "extraido"
HANOI = RAIZ / "Externos" / "Hanoi"
MODELOS = RAIZ / "Modelos"
RESULTADOS = RAIZ / "Resultados"

# Todas as panorâmicas são levadas a esta largura antes de tudo: as originais têm
# ~2400-2900 px e as lesões ~60-100 px, que viram ~30-50 px aqui.
LARGURA = 1400


def ler_cinza(caminho):
    # imdecode lida com acentos no caminho (cv2.imread não, no Windows)
    dados = np.fromfile(str(caminho), np.uint8)
    img = cv2.imdecode(dados, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"não consegui ler {caminho}")
    return img


def normalizar(img, largura=None):
    """Redimensiona para `largura` (padrão LARGURA) e equaliza o contraste (CLAHE) para aproximar aparelhos diferentes."""
    largura = largura or LARGURA
    escala = largura / img.shape[1]
    interp = cv2.INTER_AREA if escala <= 1 else cv2.INTER_CUBIC
    img = cv2.resize(img, (largura, round(img.shape[0] * escala)), interpolation=interp)
    img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    return img, escala


def redimensionar_mascara(m, forma):
    return cv2.resize(m, (forma[1], forma[0]), interpolation=cv2.INTER_NEAREST) > 0


NOMES_ATRIBUTOS = []
VERSAO_ATRIBUTOS = 2  # muda quando atributos() muda, para invalidar amostras em cache


def deslocar(A, dy, dx):
    """Valor de A no ponto (y+dy, x+dx), repetindo a borda fora da imagem."""
    p = max(abs(dy), abs(dx))
    Ap = np.pad(A, p, mode="edge")
    h, w = A.shape
    return Ap[p + dy:p + dy + h, p + dx:p + dx + w]


def atributos(img):
    """Pilha (H, W, F) de atributos por pixel, no estilo ilastik/Weka.

    A lesão periapical é uma área mais escura que o osso ao redor, perto da ponta
    da raiz: por isso brilho em várias escalas, contraste com a vizinhança (DoG),
    bordas, forma local (Hessiana), textura e posição na imagem.
    """
    I = img.astype(np.float32) / 255.0
    h, w = I.shape
    camadas, nomes = [I], ["brilho"]
    G = {s: cv2.GaussianBlur(I, (0, 0), s) for s in (1, 2, 4, 8, 16, 32)}
    for s in (1, 2, 4, 8, 16):
        camadas.append(G[s]); nomes.append(f"gauss_{s}")
    for a, b in ((1, 4), (2, 8), (4, 16), (8, 32)):
        camadas.append(G[a] - G[b]); nomes.append(f"dog_{a}_{b}")
    for s in (1, 2, 4, 8):
        gx = cv2.Sobel(G[s], cv2.CV_32F, 1, 0)
        gy = cv2.Sobel(G[s], cv2.CV_32F, 0, 1)
        camadas.append(np.sqrt(gx * gx + gy * gy)); nomes.append(f"gradiente_{s}")
    for s in (2, 4, 8):
        H = hessian_matrix(I, sigma=s, order="rc", use_gaussian_derivatives=False)
        l1, l2 = hessian_matrix_eigvals(H)
        camadas += [l1.astype(np.float32), l2.astype(np.float32)]
        nomes += [f"hessiana_{s}_l1", f"hessiana_{s}_l2"]
    for s in (4, 8):
        m2 = cv2.GaussianBlur(I * I, (0, 0), s)
        camadas.append(np.sqrt(np.maximum(m2 - G[s] ** 2, 0))); nomes.append(f"desvio_{s}")
    # Contexto: a lesão fica na ponta de uma raiz, então o brilho alguns pixels
    # acima/abaixo/ao lado (raiz clara? osso? ar?) diz muito sobre o pixel central.
    for s in (4, 8):
        for dy, dx in ((-40, 0), (-20, 0), (20, 0), (40, 0), (0, -25), (0, 25)):
            camadas.append(deslocar(G[s], dy, dx) - G[s]); nomes.append(f"contexto_{s}_{dy}_{dx}")
    camadas.append(G[32] - cv2.GaussianBlur(I, (0, 0), 64)); nomes.append("dog_32_64")
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    camadas += [xx / w, yy / h, np.abs(xx / w - 0.5)]
    nomes += ["pos_x", "pos_y", "dist_centro_x"]
    if not NOMES_ATRIBUTOS:
        NOMES_ATRIBUTOS.extend(nomes)
    return np.stack(camadas, axis=-1)


# ---------------------------------------------------------------- conjuntos

def nossas_imagens():
    """[(id, img_normalizada, mascara_bool)] das 20 panorâmicas marcadas do PDF."""
    saida = []
    for f in sorted((NOSSAS / "imagens").glob("*.png")):
        img, _ = normalizar(ler_cinza(f))
        m = redimensionar_mascara(ler_cinza(NOSSAS / "mascaras" / f.name), img.shape)
        saida.append((f.stem, img, m))
    return saida


def caixas_hanoi(nome):
    r = ET.parse(HANOI / "anotacoes" / "Image Annots" / f"{nome}.xml").getroot()
    caixas = []
    for o in r.iter("object"):
        b = o.find("bndbox")
        caixas.append(tuple(float(b.findtext(t)) for t in ("xmin", "ymin", "xmax", "ymax")))
    return caixas


def imagem_hanoi(nome, largura=None):
    """Imagem normalizada, máscara (elipse dentro de cada caixa) e zona incerta.

    A caixa é justa em torno da lesão, então a elipse inscrita é uma boa
    aproximação. A faixa entre a elipse e a caixa ampliada é marcada como
    incerta e não entra no treino nem como lesão nem como fundo.
    """
    img, esc = normalizar(ler_cinza(HANOI / "imagens" / "Original JPG Images" / f"{nome}.jpg"), largura)
    mascara = np.zeros(img.shape, np.uint8)
    incerta = np.zeros(img.shape, np.uint8)
    for x0, y0, x1, y1 in caixas_hanoi(nome):
        cx, cy = (x0 + x1) / 2 * esc, (y0 + y1) / 2 * esc
        rx, ry = (x1 - x0) / 2 * esc, (y1 - y0) / 2 * esc
        cv2.ellipse(incerta, (round(cx), round(cy)), (round(rx * 1.5), round(ry * 1.5)), 0, 0, 360, 1, -1)
        cv2.ellipse(mascara, (round(cx), round(cy)), (round(rx * 0.85), round(ry * 0.85)), 0, 0, 360, 1, -1)
    return img, mascara > 0, (incerta > 0) & (mascara == 0)


def nomes_hanoi():
    return sorted(p.stem for p in (HANOI / "imagens" / "Original JPG Images").glob("*.jpg"))


# ---------------------------------------------------------------- amostragem

def amostrar(F, mascara, incerta, n_pos, n_neg, rng):
    """Sorteia pixels de lesão e de fundo. Metade do fundo vem da faixa dos dentes,
    onde estão as confusões difíceis (raízes, espaços entre dentes)."""
    h, w = mascara.shape
    pos = np.flatnonzero(mascara)
    neg_ok = ~mascara & ~incerta
    faixa = np.zeros_like(neg_ok)
    faixa[int(0.25 * h):int(0.85 * h), int(0.15 * w):int(0.85 * w)] = True
    neg_todos = np.flatnonzero(neg_ok)
    neg_faixa = np.flatnonzero(neg_ok & faixa)
    escolha = lambda arr, n: rng.choice(arr, min(n, len(arr)), replace=False) if len(arr) else arr
    idx_pos = escolha(pos, n_pos)
    idx_neg = np.concatenate([escolha(neg_todos, n_neg // 2), escolha(neg_faixa, n_neg - n_neg // 2)])
    Fp = F.reshape(-1, F.shape[-1])
    X = np.concatenate([Fp[idx_pos], Fp[idx_neg]])
    y = np.concatenate([np.ones(len(idx_pos), np.uint8), np.zeros(len(idx_neg), np.uint8)])
    return X, y


# ---------------------------------------------------------------- predição

def prever_probabilidade(modelo, img, lote=200_000, devolver_atributos=False):
    F = atributos(img)
    Fp = F.reshape(-1, F.shape[-1])
    p = np.concatenate([modelo.predict_proba(Fp[i:i + lote])[:, 1] for i in range(0, len(Fp), lote)])
    p = p.reshape(img.shape).astype(np.float32)
    return (p, F) if devolver_atributos else p


def segmentar(prob, limiar, area_min):
    """Probabilidade -> máscara: suaviza, corta no limiar e descarta manchas pequenas."""
    p = cv2.GaussianBlur(prob, (0, 0), 2)
    m = (p >= limiar).astype(np.uint8)
    n, rot, st, _ = cv2.connectedComponentsWithStats(m)
    manter = np.zeros(n, bool)
    manter[1:] = st[1:, cv2.CC_STAT_AREA] >= area_min
    return manter[rot]


# ---------------------------------------------------------------- métricas

FRACAO_ACERTO = 0.2


def avaliar(pred, verdade):
    """Por imagem: acertos de pixel e de lesão.

    Uma mancha prevista só é acerto se pelo menos FRACAO_ACERTO dela cair sobre
    lesão de verdade; assim uma mancha enorme que só encosta na lesão não conta.
    Uma lesão verdadeira é achada se alguma mancha correta a toca.
    """
    inter = (pred & verdade).sum()
    nv, rv = cv2.connectedComponents(verdade.astype(np.uint8))
    np_, rp = cv2.connectedComponents(pred.astype(np.uint8))
    area = np.bincount(rp.ravel(), minlength=np_)
    sobre = np.bincount(rp[verdade], minlength=np_)
    corretas = np.flatnonzero(sobre >= FRACAO_ACERTO * np.maximum(area, 1))
    corretas = corretas[corretas > 0]
    achadas = len(set(np.unique(rv[np.isin(rp, corretas)])) - {0})
    return dict(inter=int(inter), soma=int(pred.sum() + verdade.sum()),
                lesoes=nv - 1, lesoes_achadas=achadas,
                previstas=np_ - 1, previstas_corretas=len(corretas))


def resumir(linhas):
    s = {k: sum(l[k] for l in linhas) for k in linhas[0]}
    return dict(
        dice=2 * s["inter"] / max(s["soma"], 1),
        sensibilidade=s["lesoes_achadas"] / max(s["lesoes"], 1),
        precisao=s["previstas_corretas"] / max(s["previstas"], 1),
        falsos_por_imagem=(s["previstas"] - s["previstas_corretas"]) / len(linhas),
    )

In [ ]:
%%writefile /content/scripts/unet.py
"""U-Net com codificador ResNet34 (torchvision), usada no Colab para treinar e aqui para prever.

Só depende de torch e torchvision, para o mesmo arquivo rodar nos dois lugares.
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

MEDIA = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DESVIO = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class Bloco(nn.Sequential):
    def __init__(self, entrada, saida):
        super().__init__(
            nn.Conv2d(entrada, saida, 3, padding=1, bias=False), nn.BatchNorm2d(saida), nn.ReLU(inplace=True),
            nn.Conv2d(saida, saida, 3, padding=1, bias=False), nn.BatchNorm2d(saida), nn.ReLU(inplace=True))


class UNetResNet34(nn.Module):
    def __init__(self, pre_treinada=False):
        super().__init__()
        r = torchvision.models.resnet34(weights="IMAGENET1K_V1" if pre_treinada else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)                   # 1/2, 64
        self.pool = r.maxpool
        self.c1, self.c2, self.c3, self.c4 = r.layer1, r.layer2, r.layer3, r.layer4  # 1/4 .. 1/32
        self.d4 = Bloco(512 + 256, 256)
        self.d3 = Bloco(256 + 128, 128)
        self.d2 = Bloco(128 + 64, 64)
        self.d1 = Bloco(64 + 64, 64)
        self.d0 = Bloco(64, 32)
        self.saida = nn.Conv2d(32, 1, 1)
        self.register_buffer("media", MEDIA.clone(), persistent=False)
        self.register_buffer("desvio", DESVIO.clone(), persistent=False)

    def forward(self, x):
        """x: (N, 1, H, W) em [0, 1], H e W múltiplos de 32. Devolve logits (N, 1, H, W)."""
        x = (x.repeat(1, 3, 1, 1) - self.media) / self.desvio
        s1 = self.stem(x)
        s2 = self.c1(self.pool(s1))
        s3 = self.c2(s2)
        s4 = self.c3(s3)
        s5 = self.c4(s4)
        sobe = lambda t: F.interpolate(t, scale_factor=2, mode="bilinear", align_corners=False)
        d = self.d4(torch.cat([sobe(s5), s4], 1))
        d = self.d3(torch.cat([sobe(d), s3], 1))
        d = self.d2(torch.cat([sobe(d), s2], 1))
        d = self.d1(torch.cat([sobe(d), s1], 1))
        return self.saida(self.d0(sobe(d)))


def carregar(caminho, dispositivo="cpu"):
    ck = torch.load(caminho, map_location=dispositivo, weights_only=False)
    m = UNetResNet34()
    m.load_state_dict(ck["modelo"])
    return m.to(dispositivo).eval(), ck.get("meta", {})


@torch.no_grad()
def prever(modelo, img, espelhar=True):
    """Mapa de probabilidade (H, W) para uma imagem uint8 já normalizada (comum.normalizar).

    Com espelhar=True faz a média com a previsão da imagem espelhada (a panorâmica
    é quase simétrica), o que costuma estabilizar o resultado.
    """
    h, w = img.shape
    H, W = -(-h // 32) * 32, -(-w // 32) * 32
    x = np.pad(img, ((0, H - h), (0, W - w)), mode="reflect").astype(np.float32) / 255.0
    disp = next(modelo.parameters()).device
    t = torch.from_numpy(x)[None, None].to(disp)
    p = torch.sigmoid(modelo(t))
    if espelhar:
        p = (p + torch.sigmoid(modelo(t.flip(-1))).flip(-1)) / 2
    return p[0, 0, :h, :w].float().cpu().numpy()

In [ ]:
%%writefile /content/scripts/treino_unet.py
"""Treina a U-Net com as panorâmicas de Hanói (feito para o Google Colab com GPU).

Usa o mesmo pré-processamento do Random Forest (comum.normalizar) e as mesmas
máscaras: elipse dentro de cada caixa = lesão, anel em volta = ignorado na perda.
Salva checkpoints em --saida a cada validação; com --continuar retoma de lá.

Versão 2 (lesões pequenas): --largura 2000 treina em resolução maior e --perda tversky
penaliza mais deixar lesão passar do que marcar a mais.

Uso: python treino_unet.py --saida PASTA [--iteracoes 6000] [--lote 8] [--largura 1400]
                           [--perda dice|tversky] [--continuar]
"""
import argparse
import math
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from joblib import Parallel, delayed
from torch.utils.data import DataLoader, Dataset

import comum as c
from unet import UNetResNet34, prever

RECORTE = (512, 768)  # altura, largura dos recortes de treino
IGNORAR = 2


def preparar_um(nome, pasta, largura):
    destino = pasta / f"{nome}_img.png"
    if destino.exists():
        return
    img, m, inc = c.imagem_hanoi(nome, largura)
    rot = np.zeros(img.shape, np.uint8)
    rot[inc] = IGNORAR
    rot[m] = 1
    cv2.imwrite(str(pasta / f"{nome}_rot.png"), rot)
    cv2.imwrite(str(destino), img)


def preparar(nomes, pasta, workers, largura):
    pasta.mkdir(parents=True, exist_ok=True)
    t = time.time()
    Parallel(n_jobs=workers)(delayed(preparar_um)(nm, pasta, largura) for nm in nomes)
    print(f"Pré-processamento: {len(nomes)} imagens em {time.time() - t:.0f}s", flush=True)


class Recortes(Dataset):
    """Recortes aleatórios; metade centrada numa lesão, para ela não ser rara demais."""

    def __init__(self, nomes, pasta, n_amostras, semente=0):
        self.nomes, self.pasta, self.n = nomes, pasta, n_amostras
        self.semente = semente

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng(self.semente * 1_000_003 + i)
        nm = self.nomes[rng.integers(len(self.nomes))]
        img = cv2.imread(str(self.pasta / f"{nm}_img.png"), cv2.IMREAD_GRAYSCALE)
        rot = cv2.imread(str(self.pasta / f"{nm}_rot.png"), cv2.IMREAD_GRAYSCALE)
        ch, cw = RECORTE
        if img.shape[0] < ch or img.shape[1] < cw:
            py, px = max(0, ch - img.shape[0]), max(0, cw - img.shape[1])
            img = np.pad(img, ((0, py), (0, px)), mode="reflect")
            rot = np.pad(rot, ((0, py), (0, px)), constant_values=IGNORAR)
        h, w = img.shape
        ys, xs = np.nonzero(rot == 1)
        if len(ys) and rng.random() < 0.5:
            k = rng.integers(len(ys))
            y0 = np.clip(ys[k] - rng.integers(ch // 4, 3 * ch // 4), 0, h - ch)
            x0 = np.clip(xs[k] - rng.integers(cw // 4, 3 * cw // 4), 0, w - cw)
        else:
            y0, x0 = rng.integers(0, h - ch + 1), rng.integers(0, w - cw + 1)
        img = img[y0:y0 + ch, x0:x0 + cw].astype(np.float32) / 255.0
        rot = rot[y0:y0 + ch, x0:x0 + cw]
        if rng.random() < 0.5:
            img, rot = img[:, ::-1], rot[:, ::-1]
        img = np.clip(img ** rng.uniform(0.7, 1.4) * rng.uniform(0.85, 1.15) + rng.uniform(-0.08, 0.08), 0, 1)
        return torch.from_numpy(img.copy())[None], torch.from_numpy(rot.copy()).long()


TVERSKY_FP, TVERSKY_FN = 0.3, 0.7  # peso de marcar a mais × peso de deixar lesão passar


def perda(logits, rot, tipo="dice"):
    """BCE + Dice (ou Tversky), só nos pixels que não estão na zona ignorada."""
    logits = logits[:, 0]
    valido = (rot != IGNORAR).float()
    alvo = (rot == 1).float()
    bce = (F.binary_cross_entropy_with_logits(logits, alvo, reduction="none") * valido).sum() / valido.sum()
    p = torch.sigmoid(logits) * valido
    vp = (p * alvo).sum()
    if tipo == "tversky":
        fp, fn = (p * (1 - alvo)).sum(), ((1 - p) * alvo).sum()
        return bce + 1 - (vp + 1) / (vp + TVERSKY_FP * fp + TVERSKY_FN * fn + 1)
    dice = 1 - (2 * vp + 1) / (p.sum() + alvo.sum() + 1)
    return bce + dice


def validar(modelo, nomes, pasta):
    modelo.eval()
    inter = soma = 0
    for nm in nomes:
        img = cv2.imread(str(pasta / f"{nm}_img.png"), cv2.IMREAD_GRAYSCALE)
        rot = cv2.imread(str(pasta / f"{nm}_rot.png"), cv2.IMREAD_GRAYSCALE)
        pred = prever(modelo, img, espelhar=False) >= 0.5
        valido = rot != IGNORAR
        inter += (pred & (rot == 1)).sum()
        soma += (pred & valido).sum() + (rot == 1).sum()
    modelo.train()
    return 2 * inter / max(soma, 1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--saida", required=True)
    ap.add_argument("--iteracoes", type=int, default=6000)
    ap.add_argument("--lote", type=int, default=8)
    ap.add_argument("--validar-cada", type=int, default=500)
    ap.add_argument("--n-val", type=int, default=200)
    ap.add_argument("--limite-imagens", type=int, default=0, help="só para testes rápidos")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--continuar", action="store_true")
    ap.add_argument("--largura", type=int, default=c.LARGURA, help="largura (px) em que as panorâmicas são treinadas")
    ap.add_argument("--perda", choices=("dice", "tversky"), default="dice")
    ap.add_argument("--sem-pre-treino", action="store_true", help="não baixa pesos ImageNet (testes)")
    args = ap.parse_args()

    disp = "cuda" if torch.cuda.is_available() else "cpu"
    saida = Path(args.saida)
    saida.mkdir(parents=True, exist_ok=True)
    nomes = c.nomes_hanoi()
    np.random.default_rng(0).shuffle(nomes)
    if args.limite_imagens:
        nomes = nomes[:args.limite_imagens]
    n_val = min(args.n_val, len(nomes) // 5)
    val, treino = nomes[:n_val], nomes[n_val:]
    pasta = c.HANOI / f"preparadas_{args.largura}"
    preparar(nomes, pasta, max(args.workers, 1), args.largura)
    print(f"Treino: {len(treino)} imagens | validação: {len(val)} | dispositivo: {disp}", flush=True)

    modelo = UNetResNet34(pre_treinada=not args.sem_pre_treino).to(disp)
    opt = torch.optim.AdamW(modelo.parameters(), lr=3e-4, weight_decay=1e-4)
    agenda = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda it: min(1, (it + 1) / 200) * 0.5 * (1 + math.cos(math.pi * it / args.iteracoes)))
    escalador = torch.amp.GradScaler(enabled=disp == "cuda")
    inicio, melhor = 0, -1.0
    ck_path = saida / "checkpoint.pt"
    if args.continuar and ck_path.exists():
        ck = torch.load(ck_path, map_location=disp, weights_only=False)
        modelo.load_state_dict(ck["modelo"]); opt.load_state_dict(ck["opt"]); agenda.load_state_dict(ck["agenda"])
        inicio, melhor = ck["iteracao"], ck["melhor"]
        print(f"Retomando da iteração {inicio} (melhor Dice de validação {melhor:.3f})", flush=True)

    dados = DataLoader(Recortes(treino, pasta, (args.iteracoes - inicio) * args.lote, semente=inicio),
                       batch_size=args.lote, num_workers=args.workers, pin_memory=disp == "cuda")
    modelo.train()
    t, acum = time.time(), 0.0
    for it, (x, y) in enumerate(dados, start=inicio + 1):
        x, y = x.to(disp, non_blocking=True), y.to(disp, non_blocking=True)
        with torch.autocast(device_type=disp, dtype=torch.float16, enabled=disp == "cuda"):
            logits = modelo(x)
        l = perda(logits.float(), y, args.perda)
        opt.zero_grad(set_to_none=True)
        escalador.scale(l).backward()
        escalador.step(opt)
        escalador.update()
        agenda.step()
        acum += l.item()
        if it % 50 == 0:
            print(f"it {it}/{args.iteracoes} perda {acum / 50:.3f} ({(time.time() - t) / 50:.2f}s/it)", flush=True)
            t, acum = time.time(), 0.0
        if it % args.validar_cada == 0 or it == args.iteracoes:
            d = validar(modelo, val, pasta)
            meta = dict(iteracao=it, dice_validacao=float(d), versao_atributos=c.VERSAO_ATRIBUTOS,
                        largura=args.largura, perda=args.perda, n_treino=len(treino))
            if d > melhor:
                melhor = d
                torch.save({"modelo": modelo.state_dict(), "meta": meta}, saida / "unet_lesoes.pt")
            torch.save({"modelo": modelo.state_dict(), "opt": opt.state_dict(), "agenda": agenda.state_dict(),
                        "iteracao": it, "melhor": melhor}, ck_path)
            print(f"== validação it {it}: Dice {d:.3f} (melhor {melhor:.3f})", flush=True)
    print(f"Fim. Melhor modelo em {saida / 'unet_lesoes.pt'}")


if __name__ == "__main__":
    main()

### 6. Baixar e extrair o conjunto de Hanói (~850 MB)

In [ ]:
%%bash
set -e
mkdir -p /content/Externos/Hanoi/imagens /content/Externos/Hanoi/anotacoes
cd /content/Externos/Hanoi
if [ ! -d "imagens/Original JPG Images" ]; then
  wget -q -O originais.rar "https://data.mendeley.com/public-files/datasets/kx52tk2ddj/files/0d016bd4-d5ac-4363-ac35-c74d87ba4022/file_downloaded"
  wget -q -O annots.rar "https://data.mendeley.com/public-files/datasets/kx52tk2ddj/files/fa6e54cc-4094-4165-9987-3a8589213696/file_downloaded"
  which unrar > /dev/null || (apt-get -qq update && apt-get -qq install -y unrar > /dev/null)
  unrar x -o+ -inul originais.rar imagens/
  unrar x -o+ -inul annots.rar anotacoes/
  rm originais.rar annots.rar
fi
echo "imagens: $(ls 'imagens/Original JPG Images' | wc -l)  anotações: $(ls 'anotacoes/Image Annots' | wc -l)"

### 7. Treinar (o log mostra a perda e, a cada 1000 iterações, o Dice na validação)

In [ ]:
CONTINUAR = False
ITERACOES = 12000
flag = '--continuar' if CONTINUAR else ''
!cd /content/scripts && python -u treino_unet.py --saida "{SAIDA}" --iteracoes {ITERACOES} --lote 8 --workers 2 --largura 2000 --perda tversky --validar-cada 1000 {flag}

### 8. Baixar o modelo treinado (salve em `ML Odonto/Modelos/unet_lesoes_v2.pt`)

In [ ]:
from google.colab import files
files.download(f'{SAIDA}/unet_lesoes.pt')